# Rail Air-Production-Unit (APU) Predictive Maintenance
## IAU Projekt – Dataset 046

**Členovia dvojice:** Samuel Ivan, Juraj Krátky
**Percentuálny podiel práce (aktuálne k prvej fáze): Samuel Ivan = 50%, Juraj Krátky = 50% 

**Cieľ projektu:** predikovať premennú `failure` (0/1) zo senzorových a prevádzkových dát železničných APU jednotiek pomocou strojového učenia. Výstupom je najlepší model + data pipeline na jeho vybudovanie.

**Dátová sada (priečinok `046/`):**
| súbor | oddeľovač | obsah | kľúč |
|---|---|---|---|
| `readings.csv` | TAB | senzorové merania v čase, obsahuje cieľovú premennú `failure` | `id_unit`, `cycle_ts` |
| `units.csv` | `,` | opis jednotiek (prevádzkové hodiny, trieda, model, depo...) | `id_unit`, `id_depot` |
| `gateways.csv` | `;` | opis komunikačných brán/senzorov | `id_unit`, `id_depot` |

Tento notebook je rozdelený podľa oficiálneho zadania (`IAU-zadanie.pdf`) na jednotlivé fázy a bodované podúlohy. Každá podúloha má markdown bunku so zadaním (vrátane bodového hodnotenia) a pod ňou prázdnu code/markdown bunku na vypracovanie.

## 0. Príprava prostredia a načítanie dát

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")


In [1]:
#preusporiadanie dat 1.2b
#previest zaznamy na rovnake datove typy 1.2b
#ciselne udaje priemer mediar atd... 1.1b
#z readings medzi sebou porovnat atributy ktore mozu medzi sebou suvisiet 1.1C

Načítanie troch zdrojových súborov (pozor na rôzne oddeľovače).

In [2]:
readings = pd.read_csv("046/readings.csv", sep="\t")
units = pd.read_csv("046/units.csv", sep=",")
gateways = pd.read_csv("046/gateways.csv", sep=";")

readings.shape, units.shape, gateways.shape


((14913, 10), (2000, 7), (16000, 11))

# Fáza 1 – Prieskumná analýza (EDA): 15 bodov

## 1.1 Základný opis dát spolu s ich charakteristikami (5b)
EDA s vizualizáciou.

### 1.1 A – Analýza štruktúr dát (1b)
Analyzujte štruktúru **súborov** (vzťahy medzi nimi, počet záznamov, typy stĺpcov) a **záznamov** (počet atribútov, dátové typy, príklady hodnôt). Zdokumentujte, ako súbory medzi sebou súvisia (spoločné kľúče `id_unit`, `id_depot`).

In [3]:
# --- Štruktúra súborov: počet záznamov, počet atribútov, dátové typy, duplicity ---
dataframes = {"readings": readings, "units": units, "gateways": gateways}

for name, df in dataframes.items():
    print(f"=== {name}.csv ===")
    print(f"Počet záznamov (riadkov):  {df.shape[0]}")
    print(f"Počet atribútov (stĺpcov): {df.shape[1]}")
    print(f"Duplicitné riadky:         {df.duplicated().sum()}")
    print("Dátové typy stĺpcov:")
    print(df.dtypes)
    print()


=== readings.csv ===
Počet záznamov (riadkov):  14913
Počet atribútov (stĺpcov): 10
Duplicitné riadky:         220
Dátové typy stĺpcov:
id_unit                   str
cycle_ts                  str
currentMotor          float64
filterDp              float64
oilTemperature        float64
reservoir_pressure    float64
rms_vibration         float64
Flowmeter             float64
dewpointDischarge     float64
failure                 int64
dtype: object

=== units.csv ===
Počet záznamov (riadkov):  2000
Počet atribútov (stĺpcov): 7
Duplicitné riadky:         0
Dátové typy stĺpcov:
serviceHours      float64
class_duty            str
codeModel             str
id_depot            int64
id_unit             int64
install_region        str
fleetCode             str
dtype: object

=== gateways.csv ===
Počet záznamov (riadkov):  16000
Počet atribútov (stĺpcov): 11
Duplicitné riadky:         0
Dátové typy stĺpcov:
id_depot              int64
signal_dbm          float64
id_gateway              str
Vendo

In [4]:
# --- Štruktúra záznamov: náhľad prvých riadkov (príklady hodnôt, formát polí) ---
for name, df in dataframes.items():
    print(f"=== {name}.csv – prvé 3 záznamy ===")
    display(df.head(3))


=== readings.csv – prvé 3 záznamy ===


,id_unit,cycle_ts,currentMotor,filterDp,oilTemperature,reservoir_pressure,rms_vibration,Flowmeter,dewpointDischarge,failure
0,U00255,01 Apr 2023,2.13647,35.27064,46.51969,8.67058,1.57706,31.55756,-0.38500,0
1,U01678,2023-10-15,2.37777,15.52975,53.89221,8.66316,2.77607,27.21814,6.24255,0
2,U01566,22 Jan 2023,4.37201,23.72927,41.83953,7.52580,3.18238,32.39756,9.00128,0


=== units.csv – prvé 3 záznamy ===


,serviceHours,class_duty,codeModel,id_depot,id_unit,install_region,fleetCode
0,7380.9,normal,APU-15,51,1,Merseburg,FE
1,NaN,N,XC-7,59,2,West Michaelborough,FB
2,24901.6,light,APU-12,35,3,Schwandorf,FE


=== gateways.csv – prvé 3 záznamy ===


,id_depot,signal_dbm,id_gateway,Vendor,install_date,id_unit,Lat,firmware_version,Lon,modelHardware,Technician
0,42,-58.7,GW-00000,Rocha,"10/30/2019, 00:00:00",463.0,40.33154,v1.2.5,-8.61180,GWX-100,Andreia Vaz
1,16,-85.8,GW-00001,Moreira,2024/11/11,1905.0,38.93701,v3.4.1,-9.07318,NODE-9,Lindsay Morales
2,37,-57.4,GW-00002,Boyer PLC,2015/11/21,48.0,40.82310,v3.4.1,-6.76567,NODE-9,Camila do Barros


In [5]:
# --- Vzťahy medzi súbormi: cez ktoré kľúče sa dajú spájať a aká je ich kardinalita ---

# jednoznačnosť kľúčov v jednotlivých súboroch
print("Unikátne id_unit:")
print(f"  readings: {readings['id_unit'].nunique()} (celkom záznamov: {len(readings)})")
print(f"  units:    {units['id_unit'].nunique()}  -> je to primárny kľúč (unikátny)? {units['id_unit'].is_unique}")
print(f"  gateways: {gateways['id_unit'].nunique()}  (NULL hodnoty: {gateways['id_unit'].isna().sum()} / {len(gateways)})")
print()
print("Unikátne id_depot:")
print(f"  units:    {units['id_depot'].nunique()}")
print(f"  gateways: {gateways['id_depot'].nunique()}  (NULL hodnoty: {gateways['id_depot'].isna().sum()})")
print()

# readings.id_unit má iný formát ("U00255") ako units.id_unit (255) -> treba zjednotiť formát pred spojením
readings_unit_num = readings['id_unit'].str.extract(r'(\d+)').astype(int)[0]
overlap_readings_units = set(readings_unit_num) & set(units['id_unit'])
print("Formát readings.id_unit:", readings['id_unit'].iloc[0], " vs. units.id_unit:", units['id_unit'].iloc[0])
print(f"Po zjednotení formátu sa prekrýva {len(overlap_readings_units)} / {readings_unit_num.nunique()} "
      f"unikátnych id_unit z readings s units.")

# priemerný počet záznamov pripadajúci na jeden kľúč = typ vzťahu (1:1 / 1:N)
print()
print(f"Priemerný počet záznamov v readings na 1 id_unit: {readings.groupby('id_unit').size().mean():.2f}  -> readings je časový rad (N:1 k units)")
print(f"Priemerný počet záznamov v gateways na 1 id_depot: {gateways.groupby('id_depot').size().mean():.2f}  -> gateways sa primárne viaže na depo (N:1), nie na jednotku")


Unikátne id_unit:
  readings: 2115 (celkom záznamov: 14913)
  units:    2000  -> je to primárny kľúč (unikátny)? True
  gateways: 2000  (NULL hodnoty: 14000 / 16000)

Unikátne id_depot:
  units:    120
  gateways: 120  (NULL hodnoty: 0)

Formát readings.id_unit: U00255  vs. units.id_unit: 1
Po zjednotení formátu sa prekrýva 1998 / 2115 unikátnych id_unit z readings s units.

Priemerný počet záznamov v readings na 1 id_unit: 7.05  -> readings je časový rad (N:1 k units)
Priemerný počet záznamov v gateways na 1 id_depot: 133.33  -> gateways sa primárne viaže na depo (N:1), nie na jednotku


**Zhrnutie štruktúry dát a vzťahov medzi súbormi:**

- **`units.csv`** (2000 riadkov, 7 atribútov) je centrálna dimenzná tabuľka – **jeden riadok = jedna APU jednotka**, `id_unit` je unikátny primárny kľúč. Žiadne duplicitné riadky.
- **`readings.csv`** (14 913 riadkov, 10 atribútov) je **časový rad meraní** – na jednu jednotku pripadá v priemere ~7 záznamov (rôzne `cycle_ts`), teda vzťah k `units` je **N:1**. Obsahuje aj 220 úplne duplicitných riadkov (treba riešiť v 1.2 A). Cieľová premenná `failure` je tu.
- **`gateways.csv`** (16 000 riadkov, 11 atribútov) sa viaže primárne cez **`id_depot`** (N:1 k units, ~133 brán na depo, bez NULL hodnôt) – stĺpec `id_unit` je v 87,5 % prázdny (14 000/16 000 NULL), takže ako spájací kľúč na úrovni jednotky je nespoľahlivý/nepoužiteľný.
- **Zistený formátový problém:** `id_unit` je v `readings` reťazec (`"U00255"`), zatiaľ čo v `units`/`gateways` je to číslo (`255`). Bez zjednotenia formátu by spojenie `readings` ↔ `units` zlyhalo (nájde sa 0 zhôd). Po normalizácii sa prekrýva 1998 z 2115 unikátnych jednotiek z `readings` – zvyšných ~117 je potrebné preskúmať v časti 1.2 (môže ísť o chybné/neznáme ID).

**Záver:** dáta je nutné integrovať spojením `readings` → `units` cez `id_unit` (po zjednotení formátu) a `units` → `gateways` cez `id_depot`; samotné `id_unit` v `gateways` sa na join nedá spoliehať.

### 1.1 B – Analýza jednotlivých atribútov (1b)
Pre **min. 10 významných atribútov** analyzujte distribúcie a základné deskriptívne štatistiky (priemer, medián, rozptyl, min/max, chýbajúce hodnoty...). Overte, či hodnoty spĺňajú očakávaný/predpísaný rozsah (napr. teploty, tlaky, percentá).

### 1.1 C – Párová analýza dát (1b)
Identifikujte vzťahy a závislosti **medzi dvojicami atribútov** (napr. korelačná matica, scatter ploty, pairploty, krížové tabuľky pre kategorické premenné).

### 1.1 D – Párová analýza: predikovaná premenná vs. ostatné (1b)
Identifikujte závislosti medzi **predikovanou premennou `failure`** a ostatnými (potenciálnymi prediktormi) – napr. boxploty podľa `failure`, korelácia s cieľovou premennou, rozdiely distribúcií.

### 1.1 E – Prvotné zamyslenie k riešeniu zadania (1b)
Zhrňte textovo: Sú niektoré atribúty medzi sebou závislé? Od ktorých atribútov zrejme závisí `failure`? Je potrebné kombinovať záznamy z viacerých súborov a ako?

_Odpoveď:_

_(doplňte textovú odpoveď sem)_

## 1.2 Identifikácia problémov, integrácia a čistenie dát (5b)

### 1.2 A – Identifikácia a riešenie problémov v dátach (2b)
Identifikujte a prvotne riešte problémy v dátach, napr.: nevhodná štruktúra dát, duplicitné záznamy, nejednotné formáty (napr. rôzne formáty dátumov v `install_date`), chýbajúce hodnoty. V dátach sa môžu nachádzať aj iné, tu nevymenované problémy.

### 1.2 B – Kontrola správnosti dát (2b)
Skontrolujte:
- či dáta obsahujú **abnormálne hodnoty** (mimo fyzikálne/logicky možného rozsahu),
- či obsahujú **nelogické dátové vzťahy**, ktoré sú následkom dátovej kolekcie a anotovania dát (napr. záporné `serviceHours`, nekonzistentné kombinácie atribútov).

### 1.2 C – Vychýlené hodnoty / outlier detection (1b)
Vyskúšajte riešiť problém outlierov **minimálne 2 technikami**:
1. odstránenie vychýlených/odľahlých pozorovaní,
2. nahradenie vychýlenej hodnoty hraničnými hodnotami rozdelenia (napr. 5. a 95. percentil – winsorizing).

## 1.3 Formulácia a štatistické overenie hypotéz o dátach (5b)

### 1.3 A – Formulácia a overenie dvoch hypotéz (4b)
Sformulujte **dve hypotézy** o dátach v kontexte predikčnej úlohy a overte ich vhodne zvoleným štatistickým testom.

Príklad formulovania: *„flowmeter má v priemere vyššiu hodnotu v stave s failure ako bez nej."*

**Hypotéza 1:** _(doplňte znenie)_

**Hypotéza 2:** _(doplňte znenie)_

### 1.3 B – Overenie štatistickej sily testov (1b)
Overte, či majú vaše štatistické testy dostatok podpory z dát (dostatočne silnú **štatistickú silu / power analysis**, napr. pomocou `statsmodels.stats.power`).

## Zhrnutie Fázy 1 – odpovede na kľúčové otázky
V tejto sekcii zhrňte odpovede na otázky zo zadania:
- Majú dáta vhodný formát pre ďalšie spracovanie?
- Aké problémy sa v nich vyskytujú?
- Nadobúdajú niektoré atribúty nekonzistentné hodnoty?
- Ako riešite tieto identifikované problémy?

_Odpoveď:_

_(doplňte zhrnutie sem)_

---
**Percentuálny podiel práce členov dvojice (Fáza 1):** _doplňte_

**Termín odovzdania Fázy 1:** nedeľa 18.10.2026 23:59 (UIS)

# Fáza 2 – Predspracovanie údajov: 15 bodov
Cieľom je pripraviť dátovú sadu (csv/tsv), kde jedno pozorovanie = jeden riadok, všetky atribúty numerické (scikit-learn vyžaduje numerické dáta), a zabezpečiť **replikovateľnosť** predspracovania na trénovacej aj testovacej množine. Ak predspracovanie zmenilo tvar/charakteristiky dát, podľa potreby zopakujte časti EDA (nebodujeme znovu, ale dokumentujte zmeny).

## 2.1 Realizácia predspracovania dát (5b)

### 2.1 A – Train/test split (1b)
Rozdeľte dáta na trénovaciu a testovaciu množinu podľa vami zvoleného pomeru. **Ďalej pracujte len s trénovacím datasetom** (testovací odložte bokom pre časť 2.3).

### 2.1 B – Transformácia do formátu vhodného pre ML (1b)
Transformujte dáta tak, aby jedno pozorovanie bolo opísané jedným riadkom a každý atribút bol v numerickom formáte (napr. integrácia súborov, encoding kategorických premenných, parsovanie dátumov). Iteratívne integrujte aj kroky čistenia dát z Fázy 1.

### 2.1 C – Škálovanie a transformácie atribútov (1b)
Aplikujte min. **2 techniky scalingu** (napr. StandardScaler, MinMaxScaler) a min. **2 techniky transformácie** (napr. log-transform, PowerTransformer/Box-Cox, one-hot encoding). Porovnajte efekty, aby ste ich vedeli vhodne skombinovať v pipeline (časť 2.3 a Fáza 3).

### 2.1 D – Zdôvodnenie volieb (2b)
Zdokumentujte a zdôvodnite rozhodnutia urobené v 2.1 (prečo tento pomer splitu, prečo tieto scalery/transformery, ako ste riešili kategorické atribúty...).

_Odpoveď:_

_(doplňte zdôvodnenie sem)_

## 2.2 Výber atribútov pre strojové učenie (5b)

### 2.2 A – Informatívnosť atribútov (3b)
Zistite, ktoré atribúty sú informatívne k predikovanej premennej `failure`, použite **min. 3 techniky** (napr. korelácia/point-biserial, mutual information, chi-kvadrát test, feature importance z stromového modelu, RFE) a porovnajte ich medzi sebou.

### 2.2 B – Zoradenie atribútov podľa dôležitosti (1b)
Zoraďte zistené atribúty podľa dôležitosti (súhrnná tabuľka/graf naprieč technikami).

### 2.2 C – Zdôvodnenie volieb (1b)
Zdôvodnite výber finálnej sady atribútov pre ďalšie modelovanie.

_Odpoveď:_

_(doplňte zdôvodnenie sem)_

## 2.3 Replikovateľnosť predspracovania (5b)

### 2.3 A – Znovupoužiteľný kód predspracovania (3b)
Upravte kód realizujúci predspracovanie trénovacej množiny tak, aby ho bolo možné **bez ďalších úprav** znovu použiť na predspracovanie testovacej množiny (napr. funkcie/triedy fitnuté len na train, aplikované na test).

### 2.3 B – Využitie sklearn.pipeline (2b)
Zostavte `sklearn.pipeline.Pipeline` (prípadne `ColumnTransformer`), ktorý spája kroky predspracovania (imputácia, scaling, transformácie, encoding) do jedného znovupoužiteľného objektu, a overte ho na testovacej množine.

---
**Percentuálny podiel práce členov dvojice (Fáza 2):** _doplňte_

**Termín odovzdania Fázy 2:** nedeľa 01.11.2026 23:59 (UIS)

# Fáza 3 – Strojové učenie: 20 bodov

> Zadanie pre túto fázu zatiaľ v `IAU-zadanie.pdf` neobsahuje detailný rozpis podúloh (len bodové hodnotenie a termín odovzdania). Štruktúru nižšie doplňte/upravte podľa pokynov zverejnených neskôr (cvičenie/UIS).

Očakávaný obsah (na základe cieľa projektu z úvodu zadania):
- natrénovanie a porovnanie viacerých modelov strojového učenia na predikciu `failure`,
- voľba a zdôvodnenie metrík vyhodnotenia (vzhľadom na prípadnú nevyváženosť tried),
- ladenie hyperparametrov (cross-validation, grid/random search),
- výber a zdôvodnenie najlepšieho modelu,
- použitie pipeline z Fázy 2 na konzistentné predspracovanie train/test dát.

**Termín odovzdania Fázy 3:** nedeľa 22.11.2026 23:59 (UIS)

# Aktivity na cvičení: 10 bodov

> Detailný rozpis aktivít zatiaľ nie je v `IAU-zadanie.pdf` uvedený (len bodové hodnotenie a termín). Doplňte podľa zadania z cvičení.

**Termín odovzdania:** nedeľa 06.12.2025 23:59 (UIS)